# 12-Qubit Google Supremacy Circuit: Tetron MBQC vs Google Amplitude Target

This adjusted notebook validates the tetron MBQC translation against Google's supplied **complex output amplitudes** rather than a simplified direct Qiskit reference circuit.

What this notebook does:

1. Load one of Google's 12-qubit random-circuit-sampling circuit files, for example `circuit_n12_m14_s0_e0_pEFGH.py`.
2. Load Google's corresponding amplitude file, for example `amplitudes_n12_m14_s0_e0_pEFGH.txt`.
3. Build a normalized 12-qubit target statevector from the text amplitudes.
4. Translate the Cirq circuit into a 24-site tetron MBQC circuit while keeping:
   - `Rz(...)` rotations as native Qiskit `rz` gates on the data qubits,
   - each calibrated `FSimGate(theta, phi)` with its actual `theta` and `phi` values.
5. Run the MBQC circuit with a statevector simulator, including mid-circuit measurements and feed-forward.
6. Compare the logical/data subsystem of the 24-qubit MBQC state to Google's 12-qubit amplitude target:

$$
F=\langle \psi_{\rm Google}|\rho_{\rm MBQC,data}|\psi_{\rm Google}\rangle.
$$

Important convention:

- In Google's amplitude text file, a bitstring is ordered as `qubit1 qubit2 ... qubit12`, matching `QUBIT_ORDER`.
- In Qiskit statevector indexing, qubit 0 is the least-significant bit. Therefore, the notebook maps a Google bitstring `b0 b1 ... b11` to Qiskit's integer index using:

```python
index = int(bitstring[::-1], 2)
```

## 1. Imports and configuration

In [ ]:
import os, sys, importlib.util
from collections import OrderedDict

import numpy as np
import matplotlib.pyplot as plt
import cirq

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.compiler import transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

# This notebook is intended to live at the repo root.
REPO_ROOT  = os.path.abspath('.')
TETRON_DIR = os.path.join(REPO_ROOT, 'src', 'tetron')
GOOGLE_DIR = os.path.join(REPO_ROOT, 'google_supremacy_circuit_files')

if TETRON_DIR not in sys.path:
    sys.path.insert(0, TETRON_DIR)

from qubit_mapping import (
    grid_to_qiskit_index,
    grid_to_sq_ancilla_index,
    grid_edge_to_qiskit_indices,
    GRID_TO_LOGICAL,
)
from mbqc_translated_gates import MBQCTranslatedGates

print('Imports OK.')
print('Repo root :', REPO_ROOT)
print('Tetron dir:', TETRON_DIR)
print('Google dir:', GOOGLE_DIR)


## 2. Load Google circuit and amplitude files

In [ ]:
CIRCUIT_BASENAME   = 'circuit_n12_m14_s0_e0_pEFGH.py'
AMPLITUDE_BASENAME = 'amplitudes_n12_m14_s0_e0_pEFGH.txt'

DATA_DIR = os.path.join(REPO_ROOT, 'data', 'google_amplitudes')


def resolve_input_file(basename):
    """Find an input file in the usual repo folders or current folder."""
    candidates = [
        os.path.join(GOOGLE_DIR, basename),
        os.path.join(DATA_DIR, basename),
        os.path.join(REPO_ROOT, basename),
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        'Could not find ' + basename + '\nTried:\n' + '\n'.join(candidates)
    )


CIRCUIT_FILE   = resolve_input_file(CIRCUIT_BASENAME)
AMPLITUDE_FILE = resolve_input_file(AMPLITUDE_BASENAME)


def load_cirq_circuit(path):
    """exec() a Google circuit data file and return (QUBIT_ORDER, CIRCUIT)."""
    spec = importlib.util.spec_from_file_location('google_circuit', path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.QUBIT_ORDER, mod.CIRCUIT


QUBIT_ORDER, CIRCUIT = load_cirq_circuit(CIRCUIT_FILE)

print(f'Loaded circuit file:   {CIRCUIT_FILE}')
print(f'Loaded amplitude file: {AMPLITUDE_FILE}')
print(f'  qubits  = {len(QUBIT_ORDER)}')
print(f'  moments = {len(CIRCUIT)}')


## 3. Build Google's amplitude target state

The amplitude file has rows of the form:

```text
bitstring real_part imag_part
```

The file may contain repeated bitstrings. The repeated entries should have the same amplitude; this loader deduplicates them.

The key convention is:

```python
qiskit_index = int(google_bitstring[::-1], 2)
```

because Google's text bitstring is ordered by `QUBIT_ORDER`, while Qiskit uses qubit 0 as the least-significant statevector bit.


In [ ]:
def load_google_amplitude_target(path, n_qubits, reverse_bitstring_for_qiskit=True,
                                 duplicate_tol=1e-12, normalize=True):
    """
    Load Google's amplitude text file and build a Qiskit-order Statevector.

    File format per row:
        bitstring real_part imag_part

    Google's bitstring convention:
        bitstring[0] corresponds to QUBIT_ORDER[0],
        bitstring[1] corresponds to QUBIT_ORDER[1], etc.

    Qiskit statevector convention:
        qubit 0 is the least-significant bit in the integer basis index.

    Therefore, for a Google bitstring b0 b1 ... b_{n-1}, use
        index = int(bitstring[::-1], 2)
    when reverse_bitstring_for_qiskit=True.
    """
    dim = 2 ** n_qubits
    amps_by_bitstring = OrderedDict()
    n_rows = 0
    n_duplicates = 0

    with open(path, 'r') as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) != 3:
                raise ValueError(f'Bad row at line {line_no}: {line!r}')

            bitstring, re_s, im_s = parts
            if len(bitstring) != n_qubits:
                raise ValueError(
                    f'Line {line_no}: bitstring length {len(bitstring)} != n_qubits {n_qubits}'
                )
            if any(c not in '01' for c in bitstring):
                raise ValueError(f'Line {line_no}: invalid bitstring {bitstring!r}')

            amp = complex(float(re_s), float(im_s))
            n_rows += 1

            if bitstring in amps_by_bitstring:
                n_duplicates += 1
                if abs(amps_by_bitstring[bitstring] - amp) > duplicate_tol:
                    raise ValueError(
                        f'Line {line_no}: duplicate bitstring {bitstring} has inconsistent amplitude.\n'
                        f'  previous = {amps_by_bitstring[bitstring]}\n'
                        f'  new      = {amp}'
                    )
            else:
                amps_by_bitstring[bitstring] = amp

    if len(amps_by_bitstring) != dim:
        print(f'Warning: found {len(amps_by_bitstring)} unique bitstrings, expected {dim}.')
        print('Missing basis states will be assigned amplitude 0.')

    vec = np.zeros(dim, dtype=complex)
    for google_bitstring, amp in amps_by_bitstring.items():
        qiskit_label = google_bitstring[::-1] if reverse_bitstring_for_qiskit else google_bitstring
        idx = int(qiskit_label, 2)
        vec[idx] = amp

    norm_before = np.linalg.norm(vec)
    if normalize:
        if norm_before == 0:
            raise ValueError('Amplitude vector has zero norm.')
        vec = vec / norm_before

    info = {
        'n_rows': n_rows,
        'n_unique_bitstrings': len(amps_by_bitstring),
        'n_duplicates': n_duplicates,
        'norm_before_normalization': norm_before,
        'normalized': normalize,
        'reverse_bitstring_for_qiskit': reverse_bitstring_for_qiskit,
    }
    return Statevector(vec), info


psi_target_google, target_info = load_google_amplitude_target(
    AMPLITUDE_FILE,
    n_qubits=len(QUBIT_ORDER),
    reverse_bitstring_for_qiskit=True,
    normalize=True,
)

print('Google amplitude target:')
for k, v in target_info.items():
    print(f'  {k}: {v}')
print(f'  final norm: {np.linalg.norm(psi_target_google.data):.12f}')
print(f'  dimension : {len(psi_target_google.data)}')


## 4. Translate the Cirq circuit into a tetron MBQC circuit

Compared with the earlier simplified version, this version keeps the Google circuit details:

- `Rz(...)` rotations are kept as native Qiskit `rz` gates on the corresponding data qubits.
- Each `FSimGate(theta, phi)` uses its own calibrated `theta` and `phi` from the Cirq file.
- The single-qubit `sqrt(X)`, `sqrt(Y)`, and `sqrt(W)` gates still use the existing MBQC translation routines.

This keeps the notebook close to your previous structure but changes the target and the calibrated-gate handling.


In [ ]:
def _is_sqrt_X(g):
    return isinstance(g, cirq.XPowGate) and np.isclose(g.exponent, 0.5)


def _is_sqrt_Y(g):
    return isinstance(g, cirq.YPowGate) and np.isclose(g.exponent, 0.5)


def _is_sqrt_W(g):
    return (isinstance(g, cirq.PhasedXPowGate)
            and np.isclose(g.phase_exponent, 0.25)
            and np.isclose(g.exponent, 0.5))


def _is_rz_gate(g):
    # cirq.Rz(rads=...) is represented as a ZPowGate.
    return isinstance(g, cirq.ZPowGate)


def _rz_rads_from_cirq_zpow(g):
    """
    Convert Cirq's ZPowGate produced by cirq.Rz(rads=theta) to Qiskit rz(theta).

    For cirq.Rz(rads=theta), exponent = theta / pi. The global phase convention
    does not affect the final state comparison.
    """
    return float(np.pi * g.exponent)


def _fsim_theta_phi(g):
    """Return calibrated theta, phi from a Cirq FSimGate."""
    return float(g.theta), float(g.phi)


def count_classical_bits(cirq_circuit):
    """Total intermediate-measurement bits the MBQC translation needs."""
    n = 0
    for moment in cirq_circuit:
        for op in moment.operations:
            g = op.gate
            if   _is_sqrt_X(g):       n += MBQCTranslatedGates.N_BITS['sqrt_X']
            elif _is_sqrt_Y(g):       n += MBQCTranslatedGates.N_BITS['sqrt_Y']
            elif _is_sqrt_W(g):       n += MBQCTranslatedGates.N_BITS['sqrt_W']
            elif _is_rz_gate(g):      pass
            elif isinstance(g, cirq.FSimGate):
                theta, phi = _fsim_theta_phi(g)
                n += MBQCTranslatedGates.n_bits_two_qubit(theta, phi)
            else:
                raise ValueError(f'Unhandled gate in count_classical_bits: {g}')
    return n


def build_mbqc_tetron_circuit(qubit_order, cirq_circuit):
    """24-qubit Qiskit MBQC translation of the Cirq circuit, keeping Rz and calibrated fSim."""
    n_int  = count_classical_bits(cirq_circuit)
    n_data = len(qubit_order)

    qr    = QuantumRegister(24,     'q')
    c_int = ClassicalRegister(n_int, 'c_int')
    c_out = ClassicalRegister(n_data, 'c_out')
    qc    = QuantumCircuit(qr, c_int, c_out)

    idx = 0  # rolling offset into c_int

    for moment in cirq_circuit:
        for op in moment.operations:
            g    = op.gate
            grid = [(q.row, q.col) for q in op.qubits]

            if _is_sqrt_X(g):
                d = grid_to_qiskit_index(*grid[0])
                a = grid_to_sq_ancilla_index(*grid[0])
                MBQCTranslatedGates.gate_sqrt_X(qc, a, d, c_int, start_idx=idx)
                idx += MBQCTranslatedGates.N_BITS['sqrt_X']

            elif _is_sqrt_Y(g):
                d = grid_to_qiskit_index(*grid[0])
                a = grid_to_sq_ancilla_index(*grid[0])
                MBQCTranslatedGates.gate_sqrt_Y(qc, a, d, c_int, start_idx=idx)
                idx += MBQCTranslatedGates.N_BITS['sqrt_Y']

            elif _is_sqrt_W(g):
                d = grid_to_qiskit_index(*grid[0])
                a = grid_to_sq_ancilla_index(*grid[0])
                MBQCTranslatedGates.gate_sqrt_W(qc, a, d, c_int, start_idx=idx)
                idx += MBQCTranslatedGates.N_BITS['sqrt_W']

            elif _is_rz_gate(g):
                # Keep Google's Rz rotations. This does not consume MBQC measurement bits.
                d = grid_to_qiskit_index(*grid[0])
                qc.rz(_rz_rads_from_cirq_zpow(g), d)

            elif isinstance(g, cirq.FSimGate):
                # Keep Google's calibrated fSim angles instead of forcing (pi/2, pi/6).
                theta, phi = _fsim_theta_phi(g)
                d1, d2, anc = grid_edge_to_qiskit_indices(grid[0], grid[1])
                MBQCTranslatedGates.gate_two_qubit(
                    qc, d1, anc, d2, c_int,
                    theta=theta, phi=phi,
                    Z1=0.0, Z2=0.0, Z3=0.0, Z4=0.0,
                    start_idx=idx,
                )
                idx += MBQCTranslatedGates.n_bits_two_qubit(theta, phi)

            else:
                raise ValueError(f'Unhandled gate: {g}')

        qc.barrier()

    if idx != n_int:
        raise RuntimeError(f'Classical-bit accounting mismatch: used {idx}, allocated {n_int}.')

    for i, q in enumerate(qubit_order):
        d = grid_to_qiskit_index(q.row, q.col)
        qc.measure(d, c_out[i])

    return qc


qc_mbqc = build_mbqc_tetron_circuit(QUBIT_ORDER, CIRCUIT)
print(f'MBQC circuit: {qc_mbqc.num_qubits} qubits, '
      f'{qc_mbqc.num_clbits} clbits, depth = {qc_mbqc.depth()}')


## 5. Remove only final data measurements and prepare statevector simulation

The MBQC circuit still contains mid-circuit measurements and feed-forward corrections. One statevector run corresponds to one random MBQC measurement trajectory. If the feed-forward rules and calibrated-gate translation are correct, the final logical/data state should agree with Google's target state for every trajectory, up to numerical precision.


In [ ]:
# Remove only the final output measurements.
# This preserves the MBQC mid-circuit measurements and feed-forward conditionals.
qc_mbqc_nom = qc_mbqc.remove_final_measurements(inplace=False)

print('MBQC without final data measurements:')
print(f'  qubits = {qc_mbqc_nom.num_qubits}, clbits = {qc_mbqc_nom.num_clbits}, depth = {qc_mbqc_nom.depth()}')

# Approximate memory for one complex128 statevector.
mem_gib = (2 ** qc_mbqc_nom.num_qubits) * 16 / 1024**3
print(f'Approximate MBQC statevector memory: {mem_gib:.2f} GiB')


## 6. Helper: logical-subsystem fidelity without building a density matrix

The full MBQC statevector lives on 24 tetrons/qubits. We only compare the 12 logical/data qubits to the 12-qubit Google amplitude target.

This computes

$$
F=\langle \psi_{\rm Google}|\rho_{\rm MBQC,data}|\psi_{\rm Google}\rangle
$$

without explicitly constructing the full density matrix.


In [ ]:
# Data qubits in the 24-site tetron register, ordered to match QUBIT_ORDER.
data_qargs = [grid_to_qiskit_index(q.row, q.col) for q in QUBIT_ORDER]
print('Data qubits in 24-site MBQC register:', data_qargs)


def subsystem_fidelity_with_pure_target(full_sv, target_sv, keep_qargs):
    """
    Compute F = <target| rho_keep |target>, where rho_keep is the reduced state
    of full_sv on keep_qargs.

    This avoids building rho_full or rho_keep explicitly. It reshapes the full
    statevector as

        psi[kept_subsystem_basis, environment_basis]

    and evaluates sum_env |<target|psi_env>|^2.

    Qiskit convention:
    - qubit 0 is the least-significant bit in the statevector index.
    - The order of keep_qargs must match the qubit order of target_sv.
    """
    full_data = np.asarray(full_sv.data)
    target    = np.asarray(target_sv.data)

    n_total = int(round(np.log2(full_data.size)))
    n_keep  = len(keep_qargs)

    if full_data.size != 2 ** n_total:
        raise ValueError('full_sv length is not a power of two.')
    if target.size != 2 ** n_keep:
        raise ValueError(
            f'target state has dimension {target.size}, but keep_qargs has {n_keep} qubits.'
        )
    if len(set(keep_qargs)) != len(keep_qargs):
        raise ValueError('keep_qargs contains duplicate qubit indices.')
    if any(q < 0 or q >= n_total for q in keep_qargs):
        raise ValueError('keep_qargs contains an index outside the full statevector.')

    env_qargs = [q for q in range(n_total) if q not in keep_qargs]

    # Axis i corresponds to Qiskit qubit i when using Fortran order.
    tensor = full_data.reshape([2] * n_total, order='F')

    # Put the logical/data axes first, in the same order as the Google target.
    tensor = np.transpose(tensor, keep_qargs + env_qargs)

    # Columns are environment branches. Rows are logical/data basis states.
    psi_mat = tensor.reshape((2 ** n_keep, 2 ** (n_total - n_keep)), order='F')

    amps = target.conj() @ psi_mat
    fidelity = np.sum(np.abs(amps) ** 2)
    return float(np.real_if_close(fidelity))


## 7. Run MBQC statevector trajectories and compare to Google's amplitude target

Each trajectory samples the random intermediate MBQC measurement outcomes. A correct feed-forward implementation should give fidelity close to 1 for every trajectory.


In [ ]:
N_TRAJECTORIES = 5
SEED0 = 1234

backend_sv = AerSimulator(method='statevector')

# Save the final post-measurement state after all MBQC feed-forward operations.
qc_mbqc_sv = qc_mbqc_nom.copy()
qc_mbqc_sv.save_statevector('psi_mbqc')

mbqc_t_sv = transpile(qc_mbqc_sv, backend_sv, optimization_level=0)

fidelities = []
infidelities = []

for k in range(N_TRAJECTORIES):
    seed = SEED0 + k
    result = backend_sv.run(mbqc_t_sv, shots=1, seed_simulator=seed).result()
    psi_mbqc_full = Statevector(result.data(0)['psi_mbqc'])

    F = subsystem_fidelity_with_pure_target(
        full_sv=psi_mbqc_full,
        target_sv=psi_target_google,
        keep_qargs=data_qargs,
    )
    fidelities.append(F)
    infidelities.append(max(0.0, 1.0 - F))
    print(f'trajectory {k:02d}, seed={seed}: fidelity = {F:.12f}, infidelity = {1.0 - F:.3e}')

print('\nSummary:')
print(f'  min fidelity   = {min(fidelities):.12f}')
print(f'  mean fidelity  = {np.mean(fidelities):.12f}')
print(f'  max infidelity = {max(infidelities):.3e}')


In [ ]:
# Plot trajectory-by-trajectory infidelity.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(N_TRAJECTORIES), infidelities, marker='o')
ax.set_xlabel('MBQC measurement trajectory')
ax.set_ylabel('1 - logical fidelity')
ax.set_title('MBQC logical-state agreement with Google amplitude target')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Optional: plot Google target vs one MBQC trajectory amplitudes

This diagnostic extracts an effective logical/data state from one MBQC trajectory if the data subsystem is nearly pure, aligns the global phase to Google's target, and plots amplitudes or probabilities.


In [ ]:
def extract_logical_state_from_full_statevector(full_sv, keep_qargs, purity_tol=1e-10):
    """
    Extract a pure logical/data state from a full statevector if the data subsystem
    is not entangled with the traced-out qubits.

    Returns:
        logical_sv: complex array of length 2**len(keep_qargs)
        purity_info: dict
    """
    full_data = np.asarray(full_sv.data)
    n_total = int(round(np.log2(full_data.size)))
    keep_qargs = list(keep_qargs)
    env_qargs = [q for q in range(n_total) if q not in keep_qargs]

    tensor = full_data.reshape([2] * n_total, order='F')
    tensor = np.transpose(tensor, keep_qargs + env_qargs)

    psi_mat = tensor.reshape(
        (2 ** len(keep_qargs), 2 ** (n_total - len(keep_qargs))),
        order='F',
    )

    U, S, Vh = np.linalg.svd(psi_mat, full_matrices=False)

    purity = np.sum(S**4)
    dominant_weight = S[0] ** 2

    if 1 - dominant_weight > purity_tol:
        print('Warning: data subsystem may be mixed/entangled with ancilla.')
        print(f'Dominant Schmidt weight = {dominant_weight:.12f}')
        print(f'Purity Tr(rho_data^2)   = {purity:.12f}')

    logical_sv = U[:, 0] * S[0]
    logical_sv = logical_sv / np.linalg.norm(logical_sv)

    return logical_sv, {
        'dominant_schmidt_weight': dominant_weight,
        'purity': purity,
        'schmidt_values': S,
    }


def align_global_phase(reference_sv, target_sv):
    """Align global phase of target_sv to reference_sv."""
    overlap = np.vdot(reference_sv, target_sv)
    if np.abs(overlap) < 1e-14:
        return target_sv
    phase = overlap / np.abs(overlap)
    return target_sv * phase.conjugate()


def plot_google_vs_mbqc_amplitudes(
    psi_target_google,
    psi_mbqc_full,
    data_qargs,
    max_states=None,
    plot_type='probability',
    threshold=0.0,
    figsize=(14, 5),
):
    """
    Plot Google target and MBQC logical amplitudes side by side.

    plot_type can be: 'probability', 'real', 'imag', or 'abs'.
    """
    target = np.asarray(psi_target_google.data)

    mbqc_logical, info = extract_logical_state_from_full_statevector(
        psi_mbqc_full,
        keep_qargs=data_qargs,
    )

    mbqc_logical = align_global_phase(target, mbqc_logical)

    n_logical = int(np.log2(len(target)))

    probs_target = np.abs(target) ** 2
    probs_mbqc = np.abs(mbqc_logical) ** 2

    if plot_type == 'probability':
        y_target = probs_target
        y_mbqc = probs_mbqc
        ylabel = 'Probability'
        title = 'Google target vs MBQC logical-basis probabilities'
    elif plot_type == 'real':
        y_target = np.real(target)
        y_mbqc = np.real(mbqc_logical)
        ylabel = 'Real amplitude'
        title = 'Google target vs MBQC real amplitudes'
    elif plot_type == 'imag':
        y_target = np.imag(target)
        y_mbqc = np.imag(mbqc_logical)
        ylabel = 'Imaginary amplitude'
        title = 'Google target vs MBQC imaginary amplitudes'
    elif plot_type == 'abs':
        y_target = np.abs(target)
        y_mbqc = np.abs(mbqc_logical)
        ylabel = 'Amplitude magnitude'
        title = 'Google target vs MBQC amplitude magnitudes'
    else:
        raise ValueError("plot_type must be one of: 'probability', 'real', 'imag', 'abs'")

    indices = np.arange(len(target))
    mask = (probs_target > threshold) | (probs_mbqc > threshold)
    indices = indices[mask]

    if max_states is not None and len(indices) > max_states:
        ranking = np.argsort(-(probs_target[indices] + probs_mbqc[indices]))
        indices = indices[ranking[:max_states]]
        indices = np.sort(indices)

    # These labels are displayed in Google's qubit order, not Qiskit's reversed statevector label.
    labels = [format(i, f'0{n_logical}b')[::-1] for i in indices]

    x = np.arange(len(indices))
    width = 0.42

    plt.figure(figsize=figsize)
    plt.bar(x - width / 2, y_target[indices], width, label='Google target')
    plt.bar(x + width / 2, y_mbqc[indices], width, label='MBQC')

    plt.xticks(x, labels, rotation=90)
    plt.ylabel(ylabel)
    plt.xlabel('Basis state in Google qubit order')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

    fidelity = np.abs(np.vdot(target, mbqc_logical)) ** 2
    print(f'Logical pure-state fidelity after phase alignment: {fidelity:.12f}')
    print(f'Infidelity: {1 - fidelity:.3e}')
    print(f'Dominant Schmidt weight: {info["dominant_schmidt_weight"]:.12f}')
    print(f'Purity estimate: {info["purity"]:.12f}')

    return {
        'google_target': target,
        'mbqc_logical_aligned': mbqc_logical,
        'fidelity': fidelity,
        'purity_info': info,
    }


In [ ]:
# Example: plot one trajectory. Reuse the last psi_mbqc_full from the loop above.
# Increase max_states or lower threshold if you want to see more basis states.
plot_out = plot_google_vs_mbqc_amplitudes(
    psi_target_google=psi_target_google,
    psi_mbqc_full=psi_mbqc_full,
    data_qargs=data_qargs,
    plot_type='probability',
    max_states=80,
    threshold=1e-6,
)


## Notes

- This notebook compares the MBQC logical output directly to Google's supplied amplitude target.
- The amplitude target includes Google's `Rz(...)` rotations and calibrated `FSimGate(theta, phi)` effects.
- The MBQC translation in this notebook keeps `Rz(...)` as native Qiskit `rz` gates and passes each calibrated `theta, phi` pair into `gate_two_qubit`.
- If the fidelity is low, the first things to check are:
  1. whether `gate_two_qubit` implements the same `FSimGate(theta, phi)` convention as Cirq,
  2. whether additional Google calibration phases should be passed through the `Z1, Z2, Z3, Z4` parameters instead of as explicit `Rz` gates,
  3. whether the bitstring reversal convention is correct for this amplitude file,
  4. whether the MBQC feed-forward branch rules are correct for all seeds.
